# GrandNexus v7.8.1-rc — phase D': questions in free language, repaired (logbook 93-94)
Phase D (logbook 93) answered free questions as accurately as canonical ones, but one question about a quality absent
from memory was read as a neighbouring quality present in it, and answered "no": one asserted error, so the claim
failed. Repair: a stricter question note, and a substitution check (when the goal names something the question does not
say, the judge compares them; if they differ, the system answers "unknown").

**Step 1, debug worlds (cells 5 and 7, about 10 minutes, about $0.5):** the SAME frozen questions and YOUR SAME audit as
phase D (nothing to read again); cell 7 runs the gate with the repaired reader.
**Step 2, the measurement (cell 8, only if the gate passed):** 20 never-used worlds (seed 23), about 800 calls through the
Batch API, about $1.6, 1.5 to 3 hours, resumable (after an interruption: run cells 1-3 again, then cell 8 only; there is
no need to run cell 7 again).

Expected: cell 7 ends with `GATE ... -> PASS`; cell 8 ends with the lines V, D1, D3 and `CLAIM ...`.
If a cell prints STOP, do not go further and send its output. Run the cells in order; if the Colab assistant offers to
rewrite a cell, decline, and do not let it read the result files (they hold the measured questions). Send back the output
of cells 2 to 9 and the zip named in cell 9 (its full name).

## 1. Get exactly v7.8.1-rc

In [ ]:
# 1. Get EXACTLY GrandNexus v7.8.1-rc (the release zip, never a session save or another version; stop if it is not found)
import os, re, glob, fnmatch, subprocess, zipfile, shutil, time
VERSION, ZIP_PATTERN = 'v7.8.1-rc', 'GrandNexus_v7.8.1-rc*.zip'
CONTENT, WORK = '/content', '/content/GrandNexus'
MYDRIVE = '/content/drive/MyDrive'

def gn_version(path):
    """(git describe, has the phase D tool) of a GrandNexus folder, or (None, False)."""
    if not os.path.isdir(os.path.join(path, '.git')):
        return None, False
    d = subprocess.run('git describe --tags', shell=True, cwd=path, capture_output=True, text=True).stdout.strip()
    tr = os.path.join(path, 'grandnexus', 'llm_interface', 'translator_v5.py')
    tool = os.path.join(path, 'tools', 'living_questions.py')
    ok = all(os.path.exists(x) for x in (tr, tool, os.path.join(path, 'tools', 'colab_save.py'))) and \
         'create_with_retry' in open(tr, encoding='utf-8').read() and 'def check_substitution' in open(tool, encoding='utf-8').read()
    return d, ok

def release(name):          # a session save is named GrandNexus_<version>-<n>-g<commit>_<tag>_<date>.zip
    return fnmatch.fnmatch(name, ZIP_PATTERN) and not re.search(r'-\d+-g[0-9a-f]{7}', name)

try:
    from google.colab import drive; drive.mount(os.path.join(CONTENT, 'drive'))
except Exception as e:
    print('Drive not mounted:', e)

found = [f for f in glob.glob(os.path.join(MYDRIVE, 'GrandNexus', ZIP_PATTERN)) + glob.glob(os.path.join(MYDRIVE, ZIP_PATTERN))
         if release(os.path.basename(f))]
if not found:
    for root, dirs, names in os.walk(MYDRIVE):
        if root[len(MYDRIVE):].count(os.sep) >= 3:
            dirs[:] = []
        found += [os.path.join(root, n) for n in names if release(n)]
found.sort(key=os.path.getmtime)
if found:
    source = found[-1]
else:
    print(f'{ZIP_PATTERN} not found in My Drive: upload it now (any other zip is refused).')
    try:
        from google.colab import files
        up = next(iter(files.upload()))
    except Exception as e:
        raise SystemExit(f'STOP: no {ZIP_PATTERN} on the Drive and no upload possible ({e}).')
    if not release(os.path.basename(up)):
        raise SystemExit(f'STOP: {up} is not the release zip {ZIP_PATTERN}.')
    source = os.path.join(CONTENT, up)
print('release zip:', source)
print('sha256:', subprocess.run(['sha256sum', source], capture_output=True, text=True).stdout.split()[0][:16])

# an existing /content/GrandNexus of another version is moved aside, never reused, never deleted
d, ok = gn_version(WORK)
if os.path.exists(WORK) and not ((d or '').startswith(VERSION) and ok):
    aside = WORK + '_aside_' + time.strftime('%Y%m%d_%H%M%S')
    shutil.move(WORK, aside); print(f'existing {WORK} ({d}) moved aside to {aside}')
if not os.path.exists(WORK):
    z = zipfile.ZipFile(source)
    top = {n.split('/')[0] for n in z.namelist() if n.strip('/')}
    z.extractall(CONTENT if top == {'GrandNexus'} else WORK)
    print('extracted')
subprocess.run('git checkout -q main', shell=True, cwd=WORK)
d, ok = gn_version(WORK)
print("git describe:", d, "| phase D' tool present:", ok)
print(subprocess.run('git log --oneline -1 && git branch', shell=True, cwd=WORK, capture_output=True, text=True).stdout)
if not ((d or '').startswith(VERSION) and ok):
    raise SystemExit(f'STOP: {WORK} is {d}, not {VERSION}. Do not run the next cells.')
print(f'OK: {VERSION}')


## 2. Packages, key, pre-flight (4 tiny calls: the reader, the writer, the judge, one question read alone)

In [ ]:
# 2. Packages, key, pre-flight check: one tiny call per model, and one question read alone (under $0.02 in all)
import subprocess, sys, os, logging
WORK = '/content/GrandNexus'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic', 'clingo', 'sympy', 'networkx', 'psutil'])
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
sys.path.insert(0, WORK); sys.path.insert(0, WORK + '/tools')
logging.disable(logging.CRITICAL)
import anthropic
import grandnexus.llm_interface.translator_v5 as T
import living_questions as Q, living_llm as B
print('SDK', anthropic.__version__, '| retry waits (s):', T.DEFAULT_RETRY_WAITS)
tr = T.Translator(model=Q.READER_MODEL)
reply = tr._call(T.SYSTEM_TRANSLATE, 'TEXT:\nAll dogs bark. Rex is a dog.\n\nQUESTION (or statement to verify):\nDoes Rex bark?', 'preflight')
r = tr.receipts[-1]
print('reader  :', 'reply received' if reply.strip() else 'EMPTY REPLY', '| model:', r.model,
      f'| input {r.tokens_in} | output {r.tokens_out} | cache write {r.cache_write} | cache read {r.cache_read}')
led = Q.Ledger()
original = 'Is Hugo a plumber?'                     # not a question of the bench
free = Q.write_list(tr.client, led, Q.WRITER_MODEL, [original])[0]
print('writer  :', repr(free), '| mechanical check:', Q.check(original, free) or 'ok')
if free:
    print('judge   :', 'SAME' if Q.judge_same(tr.client, led, Q.JUDGE_MODEL, original, free) else 'DIFFERENT')
q = Q.read_question(tr, free or 'Does Hugo fix pipes for a living?', B.question_block(B.Memory(), True, 2))
lines = [l for l in q['program'].strip().splitlines() if l.strip()]
print('question:', ' | '.join(lines), '| corrective rounds:', q['rounds'])
print('a question line came back' if any(l.strip().startswith('?') for l in lines) else 'WARNING: no question line came back (send this output; the gate will tell)')
from grandnexus.nexus_logic import parse
probes = (('Does Hugo fix pipes for a living?', 'plumber(hugo)', 'kept'),             # a role named by what he does
          ('Is Paris the city Hugo calls home?', 'lives_in(hugo, paris)', 'kept'),     # a residence in other words
          ('Does Hugo have a lot of patience?', 'calm(hugo)', 'refused'))              # the substitution of logbook 93
bad = 0
for qq, gg, want in probes:
    c = Q.check_substitution(tr.client, led, Q.JUDGE_MODEL, qq, B.goal_of_program(parse(f'? {gg}.'))[1])
    got = 'kept' if c.get('kept') else 'refused'
    bad += got != want
    print(f"check   : {qq!r} read as {gg} -> {got} (expected {want})")
if bad:
    print(f'WARNING: {bad} probe(s) of the substitution check gave another verdict than expected: send this output before going on.')
print('transient errors retried during the pre-flight:', len(tr.retry_log) + len(led.retries))


## 3. Offline tests (0 calls)

In [ ]:
# 3. Offline tests (0 calls)
import subprocess, sys, glob, os
WORK = '/content/GrandNexus'
bad = []
for t in sorted(glob.glob(os.path.join(WORK, 'tests_lab', 'test_*.py'))):
    r = subprocess.run([sys.executable, '-W', 'ignore', t], cwd=WORK, capture_output=True, text=True)
    last = (r.stdout.strip().splitlines() or ['(no output)'])[-1]
    ok = r.returncode == 0 and not last.startswith('FAIL') and 'Traceback' not in r.stderr
    print(f"{'ok ' if ok else 'BAD'} {os.path.basename(t):40} {last[:70]}")
    if not ok:
        bad.append(t)
if bad:
    raise SystemExit(f'STOP: {len(bad)} offline test(s) failed; send this output.')


## 4. What both steps will cost (0 calls)

In [ ]:
# 4. Dry runs (0 calls)
import subprocess, sys
WORK = '/content/GrandNexus'
for args in (['--seed', '11', '--worlds', '5', '--gate'], ['--seed', '23', '--worlds', '20', '--batch', '--skip-nonames']):
    r = subprocess.run([sys.executable, '-W', 'ignore', 'tools/living_questions.py', *args, '--dry-run'], cwd=WORK, capture_output=True, text=True)
    print(('step 1 (gate): ' if '--gate' in args else 'step 2 (measurement): ') + r.stdout.strip()[-600:], r.stderr[-200:])


## 5. Step 1a: the debug questions and your audit of phase D, carried over (0 calls)

In [ ]:
# 5. The frozen debug set of phase D (sha e7bd960febf84a71) and Khaled's audit of it, from the Drive or the repository
import subprocess, sys, os
WORK, STATE = '/content/GrandNexus', '/content/drive/MyDrive/GrandNexus/phaseD'
os.makedirs(STATE, exist_ok=True)
sys.path.insert(0, WORK); sys.path.insert(0, WORK + '/tools')
import argparse, logging
logging.disable(logging.CRITICAL)
import living_questions as Q
a = argparse.Namespace(seed=11, worlds=5, sessions=12, questions_dir=STATE)
fz = Q.generate(a, None)                                   # loads the frozen set: no call (client not needed)
g1, audit = Q.audit_ok(fz, STATE)
print('set sha', fz['sha'][:16], '| audit:', Q.audit_text(audit), '| G1', 'PASS' if g1 else 'FAIL')
if fz['sha'][:16] != 'e7bd960febf84a71' or not g1:
    raise SystemExit('STOP: the debug set or its audit is not the one of phase D; send this output.')


## 7. Step 1c: the gate on the debug worlds, with the repaired reader
60 fact readings (plus corrective rounds), 127 question readings and a few checks, synchronous with retries (8-12
minutes, about $0.5).

In [ ]:
# 7. Step 1c: the gate of logbook 94 on the debug worlds (G1 audit carried over, G2 = D1 on the debug worlds, G3 = V)
import subprocess, sys, os
WORK, STATE = '/content/GrandNexus', '/content/drive/MyDrive/GrandNexus/phaseD'
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
cmd = [sys.executable, '-W', 'ignore', 'tools/living_questions.py', '--seed', '11', '--worlds', '5', '--gate',
       '--questions-dir', STATE, '--state-dir', STATE]
p = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' not in line:
        sys.stdout.write(line)
p.wait(); print('exit code', p.returncode)


## 8. Step 2: the pre-registered measurement of phase D' (only if the gate passed)
Never-used worlds (seed 23, 20 worlds). Their questions are written, checked and frozen first (sha printed, no text shown), then
everything is read through the Batch API: about 14 successive batches, 1.5 to 3 hours. The state is on the Drive: after
an interruption, run cells 1-3 again, then this cell; nothing is paid twice.

In [ ]:
# 8. Step 2: the pre-registered measurement of logbook 94, ONLY if the latest gate run of phase D' passed
import subprocess, sys, os, glob, json, shutil
WORK, STATE = '/content/GrandNexus', '/content/drive/MyDrive/GrandNexus/phaseD'
RES = WORK + '/tests_lab/resultats/living'
os.makedirs(RES, exist_ok=True)
for f in glob.glob(STATE + '/living_questions_D_gate_*.json'):          # gate runs saved before a runtime reset
    if not os.path.exists(os.path.join(RES, os.path.basename(f))):
        shutil.copy2(f, RES)
runs = [f for f in sorted(glob.glob(RES + '/living_questions_D_gate_*.json'))     # the file name carries the time of the run
        if json.load(open(f, encoding='utf-8'))['args'].get('question_note') == 2]  # phase D' gates only (not those of n°93)
if not runs:
    raise SystemExit("STOP: no gate run of phase D' found. Run cells 5 and 7 first.")
d = json.load(open(runs[-1], encoding='utf-8'))
gate = d.get('gate') or {}
if not (gate.get('pass') is True and d['args']['seed'] == 11 and d['args']['worlds'] == 5 and d['args']['level'] == 2
        and d['args'].get('substitution_check') is True):
    raise SystemExit(f'STOP: the latest gate run ({os.path.basename(runs[-1])}) did not pass: {gate}. No measurement. Send the output of cells 5 to 7.')
print(f"gate of phase D' passed ({len(runs)} gate run(s) of phase D' archived):", gate)
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
cmd = [sys.executable, '-W', 'ignore', 'tools/living_questions.py', '--seed', '23', '--worlds', '20', '--batch', '--skip-nonames',
       '--questions-dir', STATE, '--state-dir', STATE]
p = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' not in line:
        sys.stdout.write(line)
p.wait(); print('exit code', p.returncode)


## 9. Save to Drive

In [ ]:
# 9. Save to Drive (the repository's own tool: it zips the repository it belongs to and checks the results are inside)
import subprocess, sys, os, glob, shutil
WORK, STATE = '/content/GrandNexus', '/content/drive/MyDrive/GrandNexus/phaseD'
RES, QDIR = WORK + '/tests_lab/resultats/living', WORK + '/tests_lab/questions'
os.makedirs(RES, exist_ok=True); os.makedirs(QDIR, exist_ok=True)
for pattern, dest in (('living_questions_D_*.json', RES), ('free_questions_*.json', QDIR), ('audit_*.json', QDIR)):
    for f in glob.glob(os.path.join(STATE, pattern)):                  # what a runtime reset would have lost
        if not os.path.exists(os.path.join(dest, os.path.basename(f))):
            shutil.copy2(f, dest)
r = subprocess.run([sys.executable, 'tools/colab_save.py', '--tag', 'phaseD2', '--expect',
                    'tests_lab/resultats/living/living_questions_D_*.json', 'tests_lab/questions/*.json'],
                   cwd=WORK, capture_output=True, text=True)
print(r.stdout, r.stderr[-500:])
